# 8.1 Equipment tables

This notebook does the following:
    - Table 1: share of labs with each equipment type at baseline and endline, pooled/control/treatment

## Set-up

In [1]:
# Set-up
import pandas as pd
import numpy as np
import sys
from pathlib import Path
CODE_ROOT = Path.cwd().parents[1]
sys.path.append(str(CODE_ROOT))
import config

In [2]:
# Load data (blank cells are genuinely missing, not the string "NA")
df = pd.read_csv(
    config.CLEAN_DATA / "final_dataset.csv",
    keep_default_na=False,
    na_values=[""]
)
panel = pd.read_csv(
    config.PROCESSED_DATA / "panel_processed_5.csv",
    keep_default_na=False,
    na_values=[""]
)

## (1) Prepare data

`panel` has one row per lab per survey per equipment type, with `number` giving the count of units of that type. Sum to get total units per lab per equipment type, at BL and at EL, then join onto every matched-sample lab (filling 0 for labs with none of that type, rather than dropping them).

In [3]:
# Keep only labgroups with both BL and EL data
labgroup_counts = df.groupby("labgroupid")["survey"].nunique()
matched_labgroupids = labgroup_counts[labgroup_counts == 2].index

df_bl = df[df["labgroupid"].isin(matched_labgroupids) & (df["survey"] == "BL")].copy()
df_equip = df_bl[["labgroupid", "treated"]].copy()

panel_matched = panel[panel["labgroupid"].isin(matched_labgroupids)].copy()

In [4]:
equipment_labels = {
    "fc": "Fume Cupboards",
    "fridge": "Fridges",
    "freezer": "Freezers",
    "ult": "ULT Freezers",
    "bath": "Water Baths",
    "heater": "Block Heaters",
    "microbio": "Microbiological Safety Cabinets",
    "incubator": "CO2 Incubators",
    "glassware": "Glassware Drying Cabinets",
    "cryostat": "Cryostats",
    "it": "IT Equipment",
}

for survey in ["bl", "el"]:
    panel_survey = panel_matched[panel_matched["survey"] == survey.upper()]
    equip_counts = (
        panel_survey.groupby(["labgroupid", "equipment"])["number"]
        .sum()
        .reset_index(name="count")
    )
    for eq in equipment_labels:
        colname = f"count_{eq}_{survey}"
        eq_sub = equip_counts[equip_counts["equipment"] == eq][["labgroupid", "count"]].rename(
            columns={"count": colname}
        )
        df_equip = df_equip.merge(eq_sub, on="labgroupid", how="left")
        df_equip[colname] = df_equip[colname].fillna(0)

## (2) Equipment summary table

For each equipment type: % of labs with ≥1 unit at Baseline and at Endline, and the percentage-point change (EL % − BL %), shown Pooled, then Control only, then Treatment only.

In [5]:
decimals = 0

col1_width = "4cm"  # width of equipment col
coln_width = "1.3cm"  # width of data cols


def fmt_pct(val, d):
    if val != val:  # NaN
        return ""
    return f"${val:.{d}f}$"


def fmt_delta(val, d):
    if val != val:  # NaN
        return ""
    magnitude = f"{abs(val):.{d}f}"
    sign = r"\llap{-}" if val < 0 else " "
    return f"${sign}{magnitude}$"

In [6]:
groups = {
    "Pooled": df_equip,
    "Control": df_equip[df_equip["treated"] == 0],
    "Treatment": df_equip[df_equip["treated"] == 1],
}

n_data_cols = 9  # 3 groups x (BL, EL, Delta)
widths = [coln_width] * n_data_cols
col_spec = f"@{{}}L{{{col1_width}}}" + "".join(f"C{{{w}}}" for w in widths)

lines = []
lines.append(f"\\begin{{tabular}}{{{col_spec}}}")
lines.append(r"\hline")
lines.append(r"\addlinespace[0.2cm]")

# Top-level column groups
lines.append(
    r" & \multicolumn{3}{c}{Pooled} & \multicolumn{3}{c}{Control}"
    r" & \multicolumn{3}{c}{Treatment} \\"
)
lines.append(r"\cmidrule(lr){2-4} \cmidrule(lr){5-7} \cmidrule(lr){8-10}")

# Sub-column labels
sub_labels = ["BL (\%)", "EL (\%)", "$\Delta$ (p.p.)"] * 3
lines.append("Equipment & " + " & ".join(sub_labels) + r" \\")
lines.append(r"\hline")
lines.append(r"\addlinespace[0.2cm]")

for eq, label in equipment_labels.items():
    row_vals = []
    for group_df in groups.values():
        bl_pct = (group_df[f"count_{eq}_bl"] > 0).mean() * 100
        el_pct = (group_df[f"count_{eq}_el"] > 0).mean() * 100
        delta = el_pct - bl_pct
        row_vals += [fmt_pct(bl_pct, decimals), fmt_pct(el_pct, decimals), fmt_delta(delta, decimals)]
    lines.append(f"{label} & " + " & ".join(row_vals) + r" \\")

lines.append(r"\addlinespace[0.2cm]")
lines.append(r"\hline")
lines.append(r"\end{tabular}")

table = "\n".join(lines)

In [7]:
out_dir = config.OUTPUT / "10_Equipment_Tables"
out_dir.mkdir(parents=True, exist_ok=True)
table_path = out_dir / "equipment_table.tex"
_ = table_path.write_text(table)